# Agentic Workflow

## 학습 목표

- 단순 pipeline과 상태 기반 agentic workflow의 차이를 이해한다.
- `AgentState`와 각 workflow node가 어떤 순서로 동작하는지 살펴본다.
- `src/workflow.py`의 실제 코드와 아키텍처 문서를 연결해서 읽는다.
- 지원하는 query type별 예시를 하나씩 비교해본다.


## 개념 설명

여기서도 먼저 현재 Python 실행 환경과 runtime profile을 확인한다. 이 노트북은 state 전이(state transition), trace, optional dense retrieval까지 다루기 때문에, 어떤 환경에서 실행되고 있는지 확인하는 단계가 특히 중요하다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## Agentic workflow란 무엇인가

정적인 pipeline은 항상 같은 절차를 반복한다. 반면 이 workflow는 여전히 명시적이고 결정적(deterministic)이지만, query type에 따라 동작 모드를 바꾸고, local tool이 필요한지 판단하고, 근거 검증(grounding verification)을 거쳐 필요하면 abstain까지 선택한다. 이 노트북에서 보는 흐름은 `docs/architecture.md`에 있는 다이어그램과 정확히 대응한다.


## 구현

이제부터는 `src/workflow.py`의 실제 node들을 순서대로 따라가며 workflow를 해부한다. 각 단계는 state가 어떻게 바뀌는지, 그리고 어느 지점에서 중요한 의사결정이 일어나는지를 눈에 보이게 보여준다.


In [ ]:
import pandas as pd

from src.ingestion import build_demo_index
from src.state import AgentState, create_initial_state
from src.trace_debug import display_trace
from src.workflow import (
    classify_query_node,
    decide_tools_node,
    fallback_or_finalize_node,
    make_plan_node,
    normalize_query_node,
    retrieve_docs_node,
    run_tools_node,
    run_workflow,
    synthesize_answer_node,
    verify_grounding_node,
)

retriever = build_demo_index(persist=False)

## 상태 기반 agent 설계(Stateful agent design)

`AgentState`는 이 프로젝트의 공용 계약(contract)이다. 이 구조를 직접 보면 workflow가 각 단계에서 무엇을 알고 있는지, 왜 다음 node가 그 정보를 바탕으로 보수적으로 판단할 수 있는지 한눈에 이해할 수 있다.


In [ ]:
state_schema = pd.DataFrame(
    {
        'field': list(AgentState.__annotations__.keys()),
        'type': [str(value) for value in AgentState.__annotations__.values()],
    }
)
state_schema

## 노드(node) 개념

첫 번째 node인 `normalize_query`는 의도적으로 단순하게 만들어져 있다. 가장 작은 state transition부터 먼저 보여주면, 이후 단계들이 모두 안정된 query 문자열 위에서 동작한다는 사실을 이해하기 쉬워진다.


In [ ]:
state = create_initial_state('How many days are in the pilot window?')
normalize_query_node(state)
pd.Series({'user_query': state['user_query'], 'normalized_query': state['normalized_query']})

## 질의 분류(query classification)

classifier는 정규화된 문장을 workflow 제어 신호(control signal)로 바꾼다. 이 저장소에서는 그 신호가 다섯 가지 query type과 `requires_tools` 플래그로 표현된다.


In [ ]:
classify_query_node(state)
pd.Series({'query_type': state['query_type'], 'requires_tools': state['requires_tools']})

## 계획(planning)

여기서 planning은 숨겨진 chain-of-thought가 아니다. 사람이 읽을 수 있는 실행 단계 목록을 고르는 일에 가깝다. 그래서 면접에서 설명하기 쉽고, 실험할 때 planner만 따로 바꿔보기도 좋다.


In [ ]:
make_plan_node(state)
pd.DataFrame({'planned_step': state['plan']})

## 검색(retrieval)

workflow가 질문의 유형을 이해한 뒤에는 가장 관련 높은 문서 청크를 가져온다. 이 단계에서는 어떤 chunk가 선택되었는지뿐 아니라, 어떤 source 문서가 reasoning context 안으로 들어왔는지도 같이 봐야 한다.


In [ ]:
retrieve_docs_node(state, retriever=retriever, top_k=4)
pd.DataFrame(state['retrieved_docs'])[['chunk_id', 'source', 'score', 'text']]

## 도구(tools)

tool 단계는 두 부분으로 나뉜다. 먼저 어떤 도구가 필요한지 결정하고, 그다음 실제로 local tool을 실행한다. pilot window 질문은 날짜 해석과 계산이 모두 필요하기 때문에, synthesis 전에 tool 사용이 계획되는 흐름을 확인할 수 있다.


In [ ]:
decide_tools_node(state)
run_tools_node(state)

pd.DataFrame(state['tool_outputs']) if state['tool_outputs'] else pd.DataFrame([{'message': 'No tool outputs'}])

## 답변 합성(answer synthesis)

synthesis는 검색된 문서 근거와 tool output을 합쳐 draft answer를 만든다. 다만 이 시점의 draft는 아직 임시 결과다. verifier가 근거를 충분히 덮고 있는지 확인하기 전까지는 최종 답으로 간주하지 않는다.


In [ ]:
synthesize_answer_node(state)
pd.Series({'draft_answer': state['draft_answer'], 'citation_count': len(state['citations'])})

## 검증(verification)

verifier는 draft가 그럴듯한지만 보지 않고, 실제로 근거를 얼마나 잘 덮고 있는지 확인한다. unsupported claim과 coverage를 보는 이유는, 마지막 단계에서 답변(answer)과 abstain을 보수적으로 구분하기 위해서다.


In [ ]:
verify_grounding_node(state)
pd.Series(state['verification_result'].to_dict())

## fallback 전략

마지막 decision node는 근거 품질을 제품 동작(product behavior)으로 바꾸는 역할을 한다. 바로 이 지점에서 workflow는 unsupported confidence 대신 abstain을 선택할 수 있다.


In [ ]:
fallback_or_finalize_node(state)
pd.Series({'final_status': state['final_status'], 'final_answer': state['final_answer']})

## workflow 실행

이제 node를 하나씩 살펴봤으니, public entrypoint인 `run_workflow(...)`가 같은 흐름을 실제로 재현하는지 확인해야 한다. 동시에 지원하는 query type 다섯 가지를 한 번씩 돌려보며 workflow가 어떻게 달라지는지 비교해본다.


In [ ]:
demo_queries = [
    ('simple_lookup', 'What are the main goals of the workspace policy refresh?'),
    ('comparison', 'How is the rollout plan different from the policy refresh?'),
    ('multi_hop', 'How many days are in the pilot window?'),
    ('summary', 'Summarize the loaded documents.'),
    ('insufficient_evidence_risk', 'Who is the current CEO of the company?'),
]
workflow_runs = []
for expected_type, question in demo_queries:
    result = run_workflow(question, retriever=retriever)
    workflow_runs.append(
        {
            'expected_demo_type': expected_type,
            'predicted_type': result['query_type'],
            'requires_tools': result['requires_tools'],
            'final_status': result['final_status'],
            'trace_steps': len(result['trace']),
            'question': question,
            'final_answer': result['final_answer'],
        }
    )

demo_frame = pd.DataFrame(workflow_runs)
demo_frame

## 실행 추적(trace) 살펴보기

trace는 한 번의 실행이 왜 그런 결과를 냈는지 이해하는 가장 빠른 방법이다. SAFE REFACTOR 이후에는 node latency까지 기록되므로, 이 표는 논리 디버깅과 성능 관찰을 동시에 도와준다.


In [ ]:
happy_path = run_workflow('How many days are in the pilot window?', retriever=retriever)
display_trace(happy_path['trace'])

## 실험

이제 다섯 가지 query type 데모를 직접 비교한다. 특히 insufficient-evidence 질문은 abstain해야 하고, multi-hop 질문은 더 많은 step과 tool 사용을 동반하는 경우가 많다는 점을 눈여겨보면 좋다.


In [ ]:
demo_frame[['expected_demo_type', 'predicted_type', 'requires_tools', 'final_status', 'trace_steps', 'question']]

## 결과 해석

이 노트북의 핵심은 구조를 이해하는 데 있다. classification이 모드를 정하고, planning이 경로를 만들고, retrieval과 tools가 근거를 공급하고, verification이 최종적으로 답변 여부를 결정한다. 이 연결고리가 보이면 agentic workflow를 훨씬 설명하기 쉬워진다.


In [ ]:
demo_frame.groupby(['predicted_type', 'final_status'])['trace_steps'].mean().reset_index()

## 핵심 정리

- 이 실험을 통해 이 저장소의 agentic behavior는 **명시적 state transition** 위에서 나온다는 점을 확인했다.
- query type, plan, tool 사용 여부, trace가 모두 눈에 보이기 때문에 디버깅과 설명이 쉽다.
- abstain은 실패가 아니라, unsupported answer를 막기 위한 의도적 제품 결정이다.
- 면접에서는 "우리 시스템이 agentic한 이유"를 물으면, **stateful control, explicit planning, grounding verification, fallback policy** 네 축으로 설명하면 설득력이 높다.
